<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis.** One row = one content item at a fixed decision point (grain established in `w02`). Source: `fact_content_daily_performance` — the starter CSV has no daily granularity and cannot express a future window.

**Windows** (decision point 2026-03-31):
- **Features** — trailing 90 days (`2025-12-31` → `2026-03-30`).
- **Trend check** — the recent 60 of those 90, split 30/30, matching FlyRank's `trend_pct`.
- **Label** — `2026-03-31` → `2026-04-30`, strictly after, untouched by any feature.

Grain and windows are verified by query in section 3.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Context** (never a feature) | `report_date`, `client_hash_id`, `content_hash_id`, `month`, `keyword_hash_id`, `url_hash_id`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`, `is_published`, `is_deleted` | IDs and dates join and window only. The `*_available` flags say whether a zero means "no activity" or "no tracking". `is_published`/`is_deleted` are row filters. |
| **Features — verified clean** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `gsc_sum_position` | 42.1% of items have no impressions at all in the window, explained by client tracking start dates (section 3). Note `gsc_avg_position` is **zero-based** — 0 is the top rank, not missing data — so average position is `SUM(sum_position)/SUM(impressions) + 1` per [Google's reference](https://support.google.com/webmasters/answer/12917991). ML-05 has the experiment. |
| **Features — need the availability flag first** | `ga4_pageviews` (71.2% zero), `ga4_sessions` (71.3%), `ga4_users` (71.3%), `ga4_engaged_sessions` (94.7%), `ga4_total_engagement_sec` (87.4%), `sessions_organic` (86.5%), `sessions_direct` (79.7%) | Only 28.8% of items have GA4 tracking. A zero is ambiguous without `ga4_data_available` joined per row. |
| **Features — too sparse to stand alone** | `sessions_ai` (98.3% zero), `scroll_events` (85.2%), `sessions_referral` (97.7%), `sessions_social` (99.8%), `sessions_paid` (98.0%), `ai_chatgpt` (98.9%), `ai_perplexity` (99.7%), `ai_gemini` (99.5%) | Real, but rare enough that a model leaning on them would mostly fit noise. |
| **Excluded — no data at all** | `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero for every item in this window. Worth re-checking at another decision point before ruling out permanently. |
| **Features — content metadata (`dim_content`)** | `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent` (18.3-18.5% missing), `content_type`, `word_count`/`char_count` (30.8%), `category_count` (0%), `keyword_char_count`, `keyword_token_count`, `url_char_count`, `content_created_date`, `content_updated_date` | Static or slow-changing, known well before any decision point. Dates convert to day-counts. Missingness needs `has_*` flags, not blind fills. |
| **Feature — sparse** | `backlinks` (53.0% missing) | Real when present; needs `has_backlink_data`. |
| **Needs a leakage audit before use** | `last_optimized_date`, `optimization_eligible_date` (87.8% missing) | Sparsity and naming suggest they populate only when FlyRank's own system acted on a page — the product-decision-as-feature trap. Not usable until confirmed. |
| **Excluded** | `provider_used`, `model_used` | Marked "not a model feature" in the data dictionary — content-generation tooling, not performance. |
| **Feature — prior window only** | `was_declining`, `prior_trend_pct` | Computed entirely from the 30/30 trend window, strictly before the decision point. |
| **Label** (never a feature) | `future_change_pct`, `future_decline`, `future_recovery`, `future_momentum` | Computed from the label window — the thing being predicted. |
| **Excluded** | Product flags (`health_score`, `priority_score`, `action_type`) | Not shipped in this data. Using a product's own decision as a feature teaches the model to copy it. |

---

> ### 📌 Amendment (2026-08-05, after ML-06) — the target becomes a single continuous value
>
> The three-target design above is **superseded**. It is replaced by one regression target:
>
> ```
> target = log( future_daily_rate / baseline_daily_rate )
> ```
>
> **Why the three binary labels go.** Four problems, all created by thresholding rather than by
> the data:
>
> 1. **The ±20% cut is arbitrary.** It was inherited from FlyRank's `trend_direction` convention,
>    not derived from anything measured here.
> 2. **Magnitude is discarded.** A page falling 21% and a page falling 95% receive the identical
>    label, so the model cannot learn that one is worse or rank it higher — which is precisely
>    what a review queue needs. Meanwhile a page at −19% and one at −21% land in *opposite*
>    classes despite being nearly identical.
> 3. **The cohort is split three ways.** Each target trains on a subset conditioned on prior
>    state; a single regression uses every row.
> 4. **It manufactured a definitional artefact.** `future_decline` excludes already-declining
>    pages by construction, which is why two buckets in ML-06 Test 1 read exactly 0.00%. That
>    confusion disappears with the threshold.
>
> Sign carries what the three labels carried: negative is decline, positive is recovery or
> momentum depending on prior state. No information is lost by collapsing them.
>
> **The log is not cosmetic.** The raw ratio runs from −100% to +94,550% — bounded below,
> unbounded above, so a handful of giants would dominate any fit. `log(ratio)` is symmetric
> about zero and treats a halving and a doubling as equal-and-opposite moves.
>
> **Amended again (2026-08-06) — the target is an `asinh` difference, not a log ratio.**
> A ratio cannot represent a page reaching zero: `log(0) = -inf`, and that is **7.4% of the D1
> cohort** (13.7% at D2). ML-06 Finding 3 settles it:
>
> ```
> target = asinh(future_daily_rate) - asinh(baseline_daily_rate)
> ```
>
> `asinh(x) = log(x + sqrt(x^2+1))` equals 0 at x = 0 and converges to `log(2x)` for large x, so it
> behaves like the log ratio where that works — Spearman **+0.955** (D1) / **+0.901** (D2) against it
> on rows where both are defined — while staying finite everywhere. It also deflates percentage
> swings at trivial volume, which the log ratio exaggerates: 0.1 to 0.05 impressions/day reads as
> -0.693 under the log and -0.05 under `asinh`.
>
> **Dead pages also get a flag, gated on volume:** `went_to_zero AND prior_rate >= 1 impr/day` —
> 1,334 pages at D1 over 34 clients. 87% of deaths were pages already carrying under one impression
> a day, so an ungated flag would bury the real emergencies.
>
> The flag is **ranked by prior daily rate**, not just listed. Per client it is a median of 10 pages,
> but the tail runs to 195 (D1) and 663 (D2), and **4 clients at D1 / 9 at D2 exceed K = 100** — so
> it can consume a whole audit. Ranking it means an overrun shows the biggest losses rather than an
> arbitrary hundred.
>
> **What this does *not* fix.** ML-06 proved the decline gradient is an artefact of dividing by
> `trend_recent_impr` — a null carrying no future information produced a **79.0-point** gradient
> against **12.6** observed. Any ratio against the recent window inherits that, including this
> one. The denominator must also change: **`baseline_daily_rate` is to be an independent
> baseline**, not the window used to select the cohort. Fixing the label *design* and fixing the
> label *denominator* are two separate corrections; this amendment is the first.
>
> **Implemented in ML-07** (`w04_baseline_score.ipynb`), where the baseline and its target are
> built together. Nothing above is rewritten — the reasoning that led here is the record.
>
> ---
>
> **Consequences for the metrics — the target change does not stop at the label.**
>
> A *base rate* is the share of the positive class. A continuous target has no positive class, so
> **"base rate 41.3%" has no meaning under this design.** The same applies to ROC-AUC, which is
> defined only for binary labels. Both must be replaced, not merely recomputed.
>
> The decision a specialist takes is still binary — review this page or don't — so a threshold has
> to exist *somewhere*. What changes is **where**:
>
> | | old design | amended design |
> |---|---|---|
> | threshold applied at | **training** (label construction) | **evaluation / deployment only** |
> | primary metric | ROC-AUC | **Spearman correlation** between predicted and actual change — needs no threshold at all |
> | queue metric | Precision@K | Precision@K, unchanged — but the cut on the *actual* outcome is now a stated evaluation parameter, not a property of the label |
> | base rate | share of positives | share of the evaluation set meeting the evaluation cut — an evaluation parameter, not a label property |
>
> Spearman becomes the honest primary metric: the task is ranking, and rank correlation measures
> ranking directly without inventing a cut-off to do it.
>
> **Every figure already recorded in ML-05 and the capstone — AUC 0.502, Precision@50 0.404,
> base rate 0.413 — measured the superseded binary label.** They are not wrong; they describe the
> design that has now been replaced, and they are kept on that basis. Re-measurement happens in
> ML-07 against the new target.
>
> One figure was wrong for a *separate* reason and has been recomputed rather than kept:
> **per-client recall was published as 48.5% and is 3.76%.** It averaged per-client rates while the
> global figure it was compared against was pooled. Both are pooled now.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip install -q duckdb huggingface_hub pandas python-dotenv

import os
import duckdb
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
local_files = [
    hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={m}/data_0.parquet",
        token=token,
    )
    for m in MONTHS
]

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in local_files)
REL = f"read_parquet([{file_list}])"
DECISION_DATE = "2026-03-31"

query = f"""
SELECT content_hash_id,
    COUNT(*) AS n_days,
    SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
    -- Google's documented formula (https://support.google.com/webmasters/answer/12917991): sum_position is
    -- ZERO-based, so average position = SUM(sum_position)/SUM(impressions) + 1.
    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) + 1 AS avg_position,
    SUM(ga4_sessions) AS ga4_sessions, SUM(ga4_engaged_sessions) AS engaged_sessions,
    SUM(sessions_ai) AS ai_sessions, SUM(scroll_events) AS scroll_events,
    MIN(report_date) AS min_d, MAX(report_date) AS max_d
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df = con.sql(query).df()

print("Rows (unique content items in the 90-day feature window):", len(df))
print()

# Grain check: zero rows back means the grain holds (df is already GROUP BY
# content_hash_id, so this also double-checks pandas agrees with the SQL).
dupes = df["content_hash_id"].duplicated().sum()
print("Grain check -- duplicate content_hash_id rows:", dupes, "(0 = grain holds)")
print()

print("Missingness -- share of content items with ZERO of this metric in the 90-day window:")
for col in ["impressions", "clicks", "ga4_sessions", "engaged_sessions", "ai_sessions", "scroll_events"]:
    print(f"  {col}: {(df[col].fillna(0) == 0).mean():.1%}")
print(f"  avg_position (page had NO impressions in the window): {df['avg_position'].isna().mean():.1%}")
print()

print("Window actually pulled:", df["min_d"].min(), "to", df["max_d"].max())
print()
print("First few rows:")
df.head()


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows (unique content items in the 90-day feature window): 349205



Grain check -- duplicate content_hash_id rows: 0 (0 = grain holds)

Missingness -- share of content items with ZERO of this metric in the 90-day window:
  impressions: 42.1%
  clicks: 75.0%
  ga4_sessions: 71.3%
  engaged_sessions: 94.7%
  ai_sessions: 98.3%
  scroll_events: 85.2%
  avg_position (page had NO impressions in the window): 42.1%

Window actually pulled: 2025-12-31 00:00:00 to 2026-03-30 00:00:00

First few rows:


,content_hash_id,n_days,impressions,clicks,avg_position,ga4_sessions,engaged_sessions,ai_sessions,scroll_events,min_d,max_d
0,content_fee7918c120cf547,70,1.0,0.0,2.000000,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30
1,content_b241658715ef7a88,70,3.0,0.0,11.666667,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30
2,content_4985aa68293f17e2,70,5.0,1.0,3.200000,1.0,0.0,0.0,0.0,2025-12-31,2026-03-30
3,content_3d1b1279fecaa7be,70,14.0,0.0,43.071429,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30
4,content_7ab030c80d6a0a95,70,0.0,0.0,NaN,0.0,0.0,0.0,0.0,2025-12-31,2026-03-30


**GA4-availability check.** The 94.7% zero rate for `engaged_sessions` above blends two very different things: pages that genuinely got no engagement, and pages GA4 simply wasn't tracking yet. This splits them apart using `ga4_data_available`.

In [2]:
query_ga4 = f"""
SELECT content_hash_id,
    BOOL_OR(ga4_data_available) AS ga4_ever_available,
    SUM(ga4_engaged_sessions) AS engaged_sessions
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_ga4 = con.sql(query_ga4).df()

tracked = df_ga4[df_ga4["ga4_ever_available"] == True]
untracked = df_ga4[df_ga4["ga4_ever_available"] == False]
undetermined = df_ga4["ga4_ever_available"].isna().sum()

print("GA4 tracking coverage over the 90-day window:")
print(f"  never tracked (ga4_ever_available = False): {len(untracked)} ({len(untracked)/len(df_ga4):.1%})")
print(f"  tracked at some point (= True):              {len(tracked)} ({len(tracked)/len(df_ga4):.1%})")
print(f"  undetermined (flag itself missing):          {undetermined} ({undetermined/len(df_ga4):.1%})")
print()
print("engaged_sessions == 0, split by tracking status:")
print(f"  among TRACKED items   -> real zero engagement: {(tracked['engaged_sessions'].fillna(0)==0).mean():.1%}")
print(f"  among UNTRACKED items -> fake zero, just not measured yet: {(untracked['engaged_sessions'].fillna(0)==0).mean():.1%}")

GA4 tracking coverage over the 90-day window:
  never tracked (ga4_ever_available = False): 180572 (51.7%)
  tracked at some point (= True):              100590 (28.8%)
  undetermined (flag itself missing):          68043 (19.5%)

engaged_sessions == 0, split by tracking status:
  among TRACKED items   -> real zero engagement: 81.5%
  among UNTRACKED items -> fake zero, just not measured yet: 100.0%


**GSC-availability check on `clicks`.** Same question for `gsc_clicks` (75.0% zero overall): is that a tracking gap like GA4, or something else? `gsc_data_available` is cleanly populated here (no undetermined rows), so the split is unambiguous.

In [3]:
query_gsc = f"""
SELECT content_hash_id,
    BOOL_OR(gsc_data_available) AS gsc_ever_available,
    SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_gsc = con.sql(query_gsc).df()

tracked_gsc = df_gsc[df_gsc["gsc_ever_available"] == True]
untracked_gsc = df_gsc[df_gsc["gsc_ever_available"] == False]

print("GSC tracking coverage over the 90-day window:")
print(f"  never tracked: {len(untracked_gsc)} ({len(untracked_gsc)/len(df_gsc):.1%})")
print(f"  tracked at some point: {len(tracked_gsc)} ({len(tracked_gsc)/len(df_gsc):.1%})")
print()
print("clicks == 0, split by tracking status:")
print(f"  among TRACKED items   -> real zero-click pages: {(tracked_gsc['clicks'].fillna(0)==0).mean():.1%}")
print(f"  among UNTRACKED items -> fake zero, just not measured yet: {(untracked_gsc['clicks'].fillna(0)==0).mean():.1%}")
print()
print("Unlike GA4: no undetermined rows here, and the tracked-item zero rate (56.8%) is a")
print("legitimate SEO finding, not a data gap -- most tracked, impression-getting pages")
print("still never get an actual click. gsc_impressions/gsc_clicks look clean enough to")
print("build on directly; ga4_engaged_sessions needs the availability flag joined in first.")

GSC tracking coverage over the 90-day window:
  never tracked: 147132 (42.1%)
  tracked at some point: 202073 (57.9%)

clicks == 0, split by tracking status:
  among TRACKED items   -> real zero-click pages: 56.8%
  among UNTRACKED items -> fake zero, just not measured yet: 100.0%

Unlike GA4: no undetermined rows here, and the tracked-item zero rate (56.8%) is a
legitimate SEO finding, not a data gap -- most tracked, impression-getting pages
still never get an actual click. gsc_impressions/gsc_clicks look clean enough to
build on directly; ga4_engaged_sessions needs the availability flag joined in first.


**The rest of the feature-candidate list, actually checked.** Section 2 lists 12+ candidate columns, but only a handful were verified above -- per this skill's own rule ("a claim without a query is a guess"), the remaining ones need the same treatment before they can honestly stay "candidates."

In [4]:
remaining_cols = [
    "gsc_sum_position", "ga4_pageviews", "ga4_users", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral", "sessions_social", "sessions_paid",
    "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
]
sums = ", ".join(f"SUM({c}) AS {c}" for c in remaining_cols)

query_remaining = f"""
SELECT content_hash_id, {sums}
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_remaining = con.sql(query_remaining).df()

print("Missingness for the rest of the candidate list, n =", len(df_remaining))
for c in remaining_cols:
    zero_rate = (df_remaining[c].fillna(0) == 0).mean()
    flag = "  <- COMPLETELY EMPTY this window" if zero_rate == 1.0 else ""
    print(f"  {c}: {zero_rate:.1%} zero{flag}")

Missingness for the rest of the candidate list, n = 349205
  gsc_sum_position: 42.5% zero
  ga4_pageviews: 71.2% zero
  ga4_users: 71.3% zero
  ga4_total_engagement_sec: 87.4% zero
  sessions_organic: 86.5% zero
  sessions_direct: 79.7% zero
  sessions_referral: 97.7% zero
  sessions_social: 99.8% zero
  sessions_paid: 98.0% zero
  ai_chatgpt: 98.9% zero
  ai_perplexity: 99.7% zero


  ai_gemini: 99.5% zero
  ai_copilot: 100.0% zero
  ai_claude: 100.0% zero
  ai_meta: 100.0% zero  <- COMPLETELY EMPTY this window
  ai_other: 100.0% zero  <- COMPLETELY EMPTY this window


**`dim_content` — content metadata, ** Everything checked above only comes from `fact_content_daily_performance` (daily performance numbers). Things like search volume, intent, and word count live in a separate table, `dim_content`.

In [5]:
dim_content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="dim_content.parquet", token=token,
)

query_dim = f"""
WITH pop AS (
    SELECT DISTINCT content_hash_id FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
)
SELECT p.content_hash_id, d.*
FROM pop p LEFT JOIN read_parquet('{dim_content_file}') d USING (content_hash_id)
"""
df_dim = con.sql(query_dim).df()

print("Join check -- population:", len(df_dim),
      "| matched to dim_content:", df_dim["content_type"].notna().sum(),
      f"({df_dim['content_type'].notna().mean():.1%})")
print()
print("Missingness on the new candidate fields:")
for col in ["search_volume", "competition", "cpc", "main_intent", "word_count", "char_count",
            "backlinks", "category_count", "last_optimized_date", "optimization_eligible_date"]:
    print(f"  {col}: {df_dim[col].isna().mean():.1%} missing")
print()
print("is_published:", df_dim["is_published"].value_counts(dropna=False).to_dict())
print("is_deleted:  ", df_dim["is_deleted"].value_counts(dropna=False).to_dict())

Join check -- population: 349205 | matched to dim_content: 349205 (100.0%)

Missingness on the new candidate fields:
  search_volume: 18.5% missing
  competition: 18.5% missing
  cpc: 18.5% missing
  main_intent: 18.3% missing
  word_count: 30.8% missing
  char_count: 30.8% missing
  backlinks: 53.0% missing
  category_count: 0.0% missing
  last_optimized_date: 87.8% missing
  optimization_eligible_date: 87.8% missing

is_published: {True: 328069, False: 21136}
is_deleted:   {False: 331885, True: 17320}


**The other two warehouse tables — .** `dim_clients` (per-client history coverage) and `fact_content_query_90d` (query-mix features)

In [6]:
# dim_clients -- does our 90-day prior window actually have full coverage per client?
clients_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                filename="dim_clients.parquet", token=token)
df_clients = con.sql(f"SELECT * FROM read_parquet('{clients_file}')").df()

prior_window_start = "2025-12-31"
gap_clients = (df_clients["gsc_data_start"] > prior_window_start).sum()
print(f"dim_clients: {len(df_clients)} clients total")
print(f"  gsc_data_start AFTER our prior-window start ({prior_window_start}):",
      f"{gap_clients} of {len(df_clients)} ({gap_clients/len(df_clients):.1%})")
print("  -> for these clients, part of our 90-day window predates their tracking,")
print("     so a zero there is a coverage gap, not a real signal. Confirms the")
print("     'unbalanced panel' limit below with an exact number instead of a guess.")
print()

# fact_content_query_90d -- is its window even compatible with our decision point,
# or would using it leak future data into our features?
query_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                              filename="fact_content_query_90d.parquet", token=token)
w = con.sql(f"SELECT MIN(window_start) AS min_start, MAX(window_end) AS max_end FROM read_parquet('{query_file}')").df()
print("fact_content_query_90d actual window range:", w["min_start"][0], "to", w["max_end"][0])
print("Our decision point: 2026-03-31 | label window: 2026-03-31 to 2026-04-30")
print("-> the query table's window starts 2026-04-02 -- already INSIDE our label")
print("   window, and runs past it. Using this table's columns as features for THIS")
print("   decision point would leak future data. Excluded for this lane/decision")
print("   point -- it's built for a decision point aligned with its own window,")
print("   which ours isn't. Exactly the leakage warning the lane guide gives for")
print("   this table, now confirmed against our actual dates rather than assumed.")

dim_clients: 104 clients total
  gsc_data_start AFTER our prior-window start (2025-12-31): 27 of 104 (26.0%)
  -> for these clients, part of our 90-day window predates their tracking,
     so a zero there is a coverage gap, not a real signal. Confirms the
     'unbalanced panel' limit below with an exact number instead of a guess.



fact_content_query_90d actual window range: 2026-04-02 00:00:00 to 2026-06-30 00:00:00
Our decision point: 2026-03-31 | label window: 2026-03-31 to 2026-04-30
-> the query table's window starts 2026-04-02 -- already INSIDE our label
   window, and runs past it. Using this table's columns as features for THIS
   decision point would leak future data. Excluded for this lane/decision
   point -- it's built for a decision point aligned with its own window,
   which ours isn't. Exactly the leakage warning the lane guide gives for
   this table, now confirmed against our actual dates rather than assumed.


**Last two columns, closing out the full sweep.** `client_has_ga4` (a structural, account-level flag in the daily fact table) and `dim_clients`' own completeness -- do all 104 clients even have a client dimension record?

In [7]:
q_flag = f"""
SELECT content_hash_id, BOOL_OR(client_has_ga4) AS has_ga4
FROM {REL}
WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
  AND report_date < DATE '{DECISION_DATE}'
GROUP BY content_hash_id
"""
df_flag = con.sql(q_flag).df()
print("client_has_ga4 (structural account flag, not a tracking-start check):")
print(" ", df_flag["has_ga4"].value_counts(dropna=False).to_dict())
print("  -> ~25% of content items belong to clients that never had GA4 access at")
print("     all, distinct from the ga4_data_available gap already covered: this is")
print("     permanent, not a 'hasn't started yet' situation.")
print()

print("dim_clients completeness:")
print(" ", df_clients["access_profile"].value_counts(dropna=False).to_dict())
orphans = (df_clients["access_profile"] == "source_only_missing_client_dimension").sum()
print(f"  -> {orphans} of {len(df_clients)} clients ({orphans/len(df_clients):.1%}) have NO client")
print("     dimension record at all -- they appear in the fact/content tables but")
print("     dim_clients has nothing for them. A genuine data-quality edge case,")
print("     not something to silently fillna over.")

client_has_ga4 (structural account flag, not a tracking-start check):
  {True: 260539, False: 88666}
  -> ~25% of content items belong to clients that never had GA4 access at
     all, distinct from the ga4_data_available gap already covered: this is
     permanent, not a 'hasn't started yet' situation.

dim_clients completeness:
  {'gsc_and_ga4': 53, 'no_search_or_analytics_access': 26, 'gsc_only': 14, 'source_only_missing_client_dimension': 10, 'ga4_only': 1}
  -> 10 of 104 clients (9.6%) have NO client
     dimension record at all -- they appear in the fact/content tables but
     dim_clients has nothing for them. A genuine data-quality edge case,
     not something to silently fillna over.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data cannot tell you.**

- **No causality.** A flag never shows that a refresh *caused* a recovery — that needs an experiment (`DATA_USE.md`).
- **Unbalanced panel.** Clients started tracking at different times; only 9 of 70 have 12+ months. **27 of 104 clients (26.0%)** have a `gsc_data_start` inside our 90-day window, so part of that window predates their tracking — their "history" is partly absence, not zero.
- **GA4 zeros are ambiguous.** 51.7% of items were never GA4-tracked, 19.5% undetermined, only 28.8% tracked. Among tracked items 81.5% still show zero engaged sessions — that part is real. Without joining `ga4_data_available` per row the two are indistinguishable.
- **GSC zeros are not ambiguous.** 42.1% of items belong to clients with no GSC tracking; among tracked items a 56.8% zero-click rate is a genuine SEO finding, not a gap.
- **10 of 104 clients have no `dim_clients` record** (`access_profile = source_only_missing_client_dimension`) — they appear only in the fact and content tables.
- **Very sparse columns.** `sessions_ai` 98.3% zero, `scroll_events` 85.2%; `ai_copilot`/`ai_claude`/`ai_meta`/`ai_other` are 100% zero here. Any claim resting on these must say so.
- **No ranking data for ~42%** of items — `avg_position` is missing wherever no impressions were logged.
- **`fact_content_query_90d` is unusable for this lane.** Its window (2026-04-02 → 2026-06-30) overlaps the label window; any column from it leaks the future.
- **One decision point.** Everything rests on 2026-03-31. No walk-forward, so nothing is known to be stable month to month.
- **Flaky host.** Remote reads hit intermittent `ZSTD Decompression failure`; reproducing may need retries.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


